# TA-5：RoBERTa 情感微调（AutoDL）

在 AutoDL **JupyterLab** 中打开本笔记本（建议把仓库放在 **`/root/autodl-tmp/RSA`**）。

**数据：** 一成子集放在 **`ml/data/splits_10pct/`**（`train.csv` / `val.csv` / `test.csv`），列需含 `analysis_input_en`、`label_sentiment`（0/1/2）。全量训练时改下一格的 `DATA_SPLIT_SUBDIR` 与 `CONFIG`。

**网络：** 克隆 GitHub / 下载 Hugging Face 模型前请运行 **学术资源加速** 一格（见 [AutoDL 学术资源加速](https://www.autodl.com/docs/network_turbo/)）。

**长时间训练：** Notebook 断连可能中断内核；更稳的方式是 SSH + **`tmux`** 里直接跑 `python ml/scripts/train_sentiment.py ...`（与本笔记本命令一致）。

In [ ]:
import os

# ============ 按实例实际情况修改 ============
REPO_DIR = "/root/autodl-tmp/RSA"
DATA_SPLIT_SUBDIR = "ml/data/splits_10pct"  # 一成；全量改为 "ml/data/splits"
CONFIG = "ml/configs/train_roberta_colab_10pct.yaml"  # 全量改为 train_roberta_colab.yaml
TOKENIZED_CACHE = "/root/autodl-tmp/ml_tokenized_cache_10pct"
HF_HOME = "/root/autodl-tmp/hf_cache"
# ==========================================

os.makedirs(HF_HOME, exist_ok=True)
os.environ["HF_HOME"] = HF_HOME

## 1) 学术资源加速（GitHub / Hugging Face）

Notebook 里每个 `!` 子进程环境独立，故用下面方式把代理写入 **当前内核** 的 `os.environ`，后续 `subprocess` 会继承。

In [ ]:
import os
import subprocess

result = subprocess.run(
    'bash -c "source /etc/network_turbo && env | grep -i proxy"',
    shell=True,
    capture_output=True,
    text=True,
)
for line in result.stdout.splitlines():
    if "=" in line:
        var, value = line.split("=", 1)
        os.environ[var.strip()] = value.strip()
print("proxy env applied:", bool(result.stdout.strip()))
if result.stderr:
    print(result.stderr)

## 2) 更新代码（`git pull`）

若尚未 `git clone`，请在 SSH 终端先克隆到 `REPO_DIR`，再运行本格。

In [ ]:
import subprocess

subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], check=True)

## 3) 安装依赖

**说明：** 若已开启学术加速，`http(s)_proxy` 可能导致 pip 访问清华源时出现 **`CERTIFICATE_VERIFY_FAILED` / self-signed certificate**。本格会对 **pip 子进程** 临时去掉代理并加 `--trusted-host`；装完后训练格仍会带代理，便于拉 Hugging Face 模型。

In [ ]:
import os
import subprocess

_env = os.environ.copy()
for _k in list(_env.keys()):
    if _k.lower() in ("http_proxy", "https_proxy", "all_proxy", "HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY"):
        _env.pop(_k, None)

subprocess.run(
    [
        "pip",
        "install",
        "-q",
        "-r",
        "ml/requirements-finetune.txt",
        "-i",
        "https://pypi.tuna.tsinghua.edu.cn/simple",
        "--trusted-host",
        "pypi.tuna.tsinghua.edu.cn",
        "--trusted-host",
        "files.pythonhosted.org",
    ],
    cwd=REPO_DIR,
    env=_env,
    check=True,
)

## 4) 确认数据文件

In [ ]:
import os

split_dir = os.path.join(REPO_DIR, DATA_SPLIT_SUBDIR)
for name in ("train.csv", "val.csv", "test.csv"):
    p = os.path.join(split_dir, name)
    print(name, "OK" if os.path.isfile(p) else "MISSING", p)

## 5)（可选）检查 GPU

In [ ]:
import torch

print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 6) 训练（一成子集）

产物目录由 yaml 指定（一成：`ml/artifacts/roberta-sentiment-v0-colab-10pct/`）。**长时间**建议改用 SSH + `tmux` 跑同一条命令以免内核断开。

In [ ]:
import os
import subprocess

env = os.environ.copy()
env["HF_HOME"] = HF_HOME

cmd = [
    "python",
    "ml/scripts/train_sentiment.py",
    "--config",
    CONFIG,
    "--tokenized-cache-dir",
    TOKENIZED_CACHE,
]
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)

## 7)（可选）测试集评估

In [ ]:
import os
import subprocess

env = os.environ.copy()
env["HF_HOME"] = HF_HOME

ckpt = "ml/artifacts/roberta-sentiment-v0-colab-10pct"  # 与 CONFIG 一致
subprocess.run(
    [
        "python",
        "ml/scripts/evaluate_sentiment.py",
        "--config",
        CONFIG,
        "--checkpoint_dir",
        ckpt,
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)